
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



# Evaluation with Mosaic AI Agent Evaluation

In previous demonstrations, we utilized `mlflow` for evaluation purposes. Mosaic AI Agent Evaluation builds upon MLflow, offering additional features and enhancements. It enables the definition of custom evaluation metrics, facilitates straightforward model deployment, and provides an easy-to-use **Review App**.

**Learning Objectives:**

*By the end of this demo, you will be able to:*

- Load a model from the model registry and use it to evaluate an evaluation dataset.
- Define custom evaluation metrics.
- Deploy the model along with the Review App to gather human feedback.

## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

   - Click **More** in the drop-down.
   
   - In the **Attach to an existing compute resource** window, use the first drop-down to select your unique cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

2. Find the triangle icon to the right of your compute cluster name and click it.

3. Wait a few minutes for the cluster to start.

4. Once the cluster is running, complete the steps above to select your cluster.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need to use one of the following Databricks runtime(s): **15.4.x-cpu-ml-scala2.12**

**🚨 Pre-requisite Notice:** This notebook requires **[00 - Build-Model]($../00-Build-Model/00-Build-Model)** to create a model that will be used for this demo. **In Databricks provided lab environment this will be run before the class, which means you don't need to run it manually**.


## Classroom Setup

Install required libraries.

In [0]:
%pip install -U -qq databricks-agents databricks-sdk databricks-vectorsearch langchain==0.3.7 databricks-langchain mlflow-skinny[databricks]==3.4.0
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
databricks-feature-engineering 0.8.0 requires mlflow-skinny[databricks]<3,>=2.11.0, but you have mlflow-skinny 3.4.0 which is incompatible.
databricks-feature-engineering 0.8.0 requires protobuf<5,>=3.12.0, but you have protobuf 5.29.5 which is incompatible.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.12.0 which is incompatible.
msal 1.29.0 requires cryptography<45,>=2.5, but you have cryptography 45.0.7 which is incompatible.
oci 2.126.4 requires cryptography<43.0.0,>=3.2.1, but you have cryptography 45.0.7 which is incompatible.
pyopenssl 23.2.0 requires cryptography!=40.0.0,!=40.0.1,<42,>=38.0.0, but you have cryptography 45.0.7 which is incompatible.
tensorboard-plugin-profile 2.15.1 requires protobuf<5.0.0dev,>=3.19.6, but you have protobuf 5.29.5 which is incompatible.
tensorflow 2.1

Before starting the demo, run the provided classroom setup script. This script will define configuration variables necessary for the demo. Execute the following cell:

In [0]:
%run ../Includes/Classroom-Setup-04


The examples and models presented in this course are intended solely for demonstration and educational purposes.
 Please note that the models and prompt examples may sometimes contain offensive, inaccurate, biased, or harmful content.


Created eval_dataset with 12 evaluation items


**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

Username:          labuser12546244_1764715360@vocareum.com
Catalog Name:      dbacademy
Schema Name:       labuser12546244_1764715360
Working Directory: /Volumes/dbacademy/ops/labuser12546244_1764715360@vocareum_com
Dataset Location:  NestedNamespace (news='/Volumes/dbacademy_news/v01', arxiv='/Volumes/dbacademy_arxiv/v01')


## Demo Overview

In this demo, we will begin by reviewing **the dataset** that will be used for evaluation. Next, we will **load a RAG chain** model from the model registry and utilize it for evaluation purposes. To illustrate custom evaluation, we will define a custom metric and incorporate it into the evaluation workflow. Upon completing the evaluation, we will **deploy the model** and demonstrate how to use the integrated "Review App" to gather **human feedback**.


## Prepare Evaluation Dataset

This dataset includes sample queries and their corresponding expected responses. The expected responses are generated using synthetic data. In a real-world project, these responses would be crafted by experts.

In [0]:
display(DA.eval_dataset)

expectations,inputs
"Map(expected_response -> Symbolic planning in task and motion planning can be limited by the need for explicit primitives and constraints. Leveraging large language models can help overcome these limitations by enabling the robot to use language models for planning and execution, and by providing a way to extract and leverage knowledge from large language models to solve temporally extended tasks.)","Map(request -> What are the limitations of symbolic planning in task and motion planning, and how can leveraging large language models help overcome these limitations?)"
"Map(expected_response -> The techniques used to fine-tune transformer models for personalized code generation include fine-tuning transformer models, adopting a novel approach called Target Similarity Tuning (TST) to retrieve a small set of examples from a training bank, and utilizing these examples to train a pretrained language model. The effectiveness of these techniques is shown in the improvement in prediction accuracy and the prevention of runtime errors.)","Map(request -> What are some techniques used to fine-tune transformer models for personalized code generation, and how effective are they in improving prediction accuracy and preventing runtime errors?)"
"Map(expected_response -> The PPO-ptx model mitigates performance regressions in the few-shot setting by incorporating pre-training and fine-tuning on the downstream task. This approach allows the model to learn generalizable features and adapt to new tasks more effectively, leading to improved few-shot performance.)",Map(request -> How does the PPO-ptx model mitigate performance regressions in the few-shot setting?)
"Map(expected_response -> Successive prompting is a method for decomposing complex questions into simpler sub-questions, allowing language models to answer them more accurately. This approach was proposed by Dheeru Dua, Shivanshu Gupta, Sameer Singh, and Matt Gardner in their paper 'Successive Prompting for Decomposing Complex Questions', presented at EMNLP 2022.)",Map(request -> How can complex questions be decomposed using successive prompting?)
Map(expected_response -> Organization),"Map(request -> Which entity type in Named Entity Recognition is likely to be involved in information extraction, question answering, semantic parsing, and machine translation?)"
"Map(expected_response -> ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is used in automatic evaluation methods to evaluate the quality of machine translation. It calculates N-gram co-occurrence statistics, which are used to assess the similarity between the candidate text and the reference text. ROUGE is based on recall, whereas BLEU is based on accuracy.)",Map(request -> What is the purpose of ROUGE (Recall-Oriented Understudy for Gisting Evaluation) in automatic evaluation methods?)
"Map(expected_response -> The challenges associated with Foundation SSL in CV include the lack of a profound theory to support all kinds of tentative experiments, and further exploration has no handbook. The pretrained LM may not learn the meaning of the language, relying on corpus learning instead. The models cannot reach a better level of stability and match different downstream tasks, and the primary method is to increase data, improve computation power, and design training procedures to achieve better results. The lack of theoretical foundation, semantic understanding, and explicable exploration are the main challenges in Foundation SSL in CV.)","Map(request -> What are the challenges associated with Foundation SSL in CV, and how do they relate to the lack of theoretical foundation, semantic understanding, and explicable exploration?)"
"Map(expected_response -> ChatGPT handles factual input better than GPT-3.5, with a 21.9% increase in accuracy when the premise entails the hypothesis. This is possibly related to the preference for human feedback in ChatGPT's RLHF design during model training.)",Map(request -> How do

## Load the Model

A RAG chain has been created and registered for us. If you're interested in the code, you can explore the `00 - Build Model` folder. Please note that building RAG chains is beyond the scope of this course. For more information on these topics, you can refer to the related course, **"Generative AI Solution Development"** available on the Databricks Academy.

In [0]:
import mlflow

catalog_name = "genai_shared_catalog_03"
schema_name = f"ws_{spark.conf.get('spark.databricks.clusterUsageTags.clusterOwnerOrgId')}"

model_name= f"{catalog_name}.{schema_name}.rag_app"
model_uri = f"models:/{catalog_name}.{schema_name}.rag_app/1"
rag_model = mlflow.langchain.load_model(model_uri)

2025/12/02 22:51:45 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-9cdcb65cf6644e938014daff0d62cabc
2025/12/02 22:51:45 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.


In [0]:
rag_model

{
  question: RunnableLambda(itemgetter('messages'))
            | RunnableLambda(extract_user_query_string),
  context: RunnableLambda(itemgetter('messages'))
           | RunnableLambda(extract_user_query_string)
           | RunnableLambda(vector_search_as_retriever)
           | RunnableLambda(format_context)
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for GENAI teaching class. You are answering questions related to Generative AI and how it impacts humans life. If the question is not related to one of these topics, kindly decline to answer. If you don't know the answer, just say that you don't know, don't try to make up an answer. Keep the answer as concise as possible. Use the following pieces of context to answer the question at the end: <context>{context}</context>")

## Model Evaluation

### Define Custom Metrics
Although the Agents Evaluation framework automatically calculates common evaluation metrics, there are instances where we may need to assess the model using custom metrics. In this section, we will define a custom metric to evaluate whether the **retrieval model** generates responses containing personally identifiable information (PII).

We can create our own evaluation metrics using prompting

**The function `make_genai_metric_from_prompt` creates a custom evaluation metric for GenAI (Generative AI) models in MLflow, based on a prompt you define. It allows you to specify a prompt template, a model to use for evaluation (such as an LLM endpoint), and metadata about the metric. This metric can then be used in MLflow's evaluation workflows to systematically assess model outputs, such as checking for PII, correctness, or other criteria, using LLM-based judgments. This helps automate and standardize the evaluation of GenAI applications**

In [0]:
from mlflow.metrics.genai import make_genai_metric_from_prompt

# Define a custom assessment to detect PII in the retrieved chunks. 
has_pii_prompt = "Your task is to determine whether the retrieved content has any PII information. This was the content: '{retrieved_context}'"

has_pii = make_genai_metric_from_prompt(
    name="has_pii",
    judge_prompt=has_pii_prompt,
    model="endpoints:/databricks-meta-llama-3-3-70b-instruct",
    metric_metadata={"assessment_type": "RETRIEVAL"},
    greater_is_better = False
)

In [0]:
has_pii.greater_is_better

False

In [0]:
# Evaluation Metric
has_pii

### Run Evaluation Test

Please note that in the code below, we are logging the evaluation process using MLflow to enable viewing the results through the MLflow UI.

In [0]:
with mlflow.start_run(run_name="rag_eval_with_agent_evaluation"):
    eval_results = mlflow.evaluate(
        data = DA.eval_df,
        model = model_uri,
        model_type = "databricks-agent",
        extra_metrics=[has_pii]
    )

/local_disk0/.ephemeral_nfs/envs/pythonEnv-dd226c41-d6b5-44be-9eac-da03541bc777/lib/python3.11/site-packages/mlflow/models/evaluation/deprecated.py:13: FutureWarning: The `mlflow.evaluate` API has been deprecated as of MLflow 3.0.0. Please use these new alternatives:

 - For traditional ML or deep learning models: Use `mlflow.models.evaluate`, which maintains full compatibility with the original `mlflow.evaluate` API.

 - For LLMs or GenAI applications: Use the new `mlflow.genai.evaluate` API, which offers enhanced features specifically designed for evaluating LLMs and GenAI applications.

  warnings.warn(


2025/12/02 22:51:51 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - databricks-vectorsearch (current: 0.63, required: databricks-vectorsearch==0.60)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2025/12/02 22:51:52 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-9cdcb65cf6644e938014daff0d62cabc
2025/12/02 22:51:52 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2025/12/02 22:51:52 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.


Evaluating:   0%|          | 0/12 [Elapsed: 00:00, Remaining: ?] 

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

<!DOCTYPE html>
 
 
 Evaluation output 
 
 
 
 
 
 
 
 
 View evaluation results.

[Trace(trace_id=tr-e1424db71f061a2fa53ee104492923a5), Trace(trace_id=tr-6e405b3875a620113ba6d62d3f3e1e22), Trace(trace_id=tr-afcfd6d06e500d06547b5bad0a322761), Trace(trace_id=tr-806292f040bc00980cf54dffd9325e92), Trace(trace_id=tr-146876806edcef9646528f686b5dc1ca), Trace(trace_id=tr-b43081d3486fc90aa91c286b64af22fd), Trace(trace_id=tr-3be6e3869b617338fa466d1d2534b2ac), Trace(trace_id=tr-928ab00b745d3f201a67d144c3a9f164), Trace(trace_id=tr-ef22330b7daac72e783fbbad11f6816e), Trace(trace_id=tr-66042b38847073da2f15c3175c693ba6)]

### Review Evaluation Results

We have two options for reviewing the evaluation results. The first option is to examine the metrics and tables directly using the results object. The second option is to review the results through the user interface (UI).

#### Review Review Metrics

In [0]:
display(eval_results.metrics)

{'has_pii/precision/mean': 1.0,
 'agent/latency_seconds/mean': 5.620666666666666,
 'safety/percentage': 1.0,
 'groundedness/percentage': 0.6666666666666666,
 'correctness/percentage': 0.25,
 'context_sufficiency/percentage': 0.5}

### Review Results via the UI

To view the results in the UI, follow these steps:

- Click on the **"View evaluation results"** tab displayed at the **Run Evaluation Test** section code block's output for a simpler method.

- Alternatively, you can navigate to "Experiments" in the left panel and locate the experiment registered with the title of this notebook.

- Click on the Run Name and View the overall metrics in the **Model Metrics** tab.

- Examine detailed results for each assessment in the **Evaluation results Preview** tab.

## Collect Human Feedback via Databricks Review App

The Databricks Review App stages the LLM in an environment where expert stakeholders can engage with it—allowing for conversations, questions, and more. This setup enables the collection of valuable feedback on your application, ensuring the quality and safety of its responses.

**Stakeholders can interact with the application bot and provide feedback on these interactions. They can also offer feedback on historical logs, curated traces, or agent outputs.**

**🚨 Important Note:**

This step is **for instructors only**. If you are using your own environment, you can comment out the cells and run them to deploy the model and access the Review App.

**⚠️ Warning: Permission Required**

If you are not an instructor and try to run this step without the required permissions, you may encounter the `PermissionDenied` error.

**How to Proceed:**
- **If you are an instructor**, after running this code, you must grant permissions to users as needed.
- **If you are not an instructor**, do **not** run this step without getting permission from an instructor. Otherwise, you will encounter a permission error and won’t be able to proceed.

In [0]:
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointStateReady, EndpointStateConfigUpdate
import mlflow
from databricks import agents

# Deploy the model with the agent framework
deployment_info = agents.deploy(
    model_name, 
    model_version=1,
    scale_to_zero=True,
    budget_policy_id=None)

# Wait for the Review App and deployed model to be ready
w = WorkspaceClient()
print("\nWaiting for endpoint to deploy.  This can take 15 - 20 minutes.", end="")

while ((w.serving_endpoints.get(deployment_info.endpoint_name).state.ready == EndpointStateReady.NOT_READY) or (w.serving_endpoints.get(deployment_info.endpoint_name).state.config_update == EndpointStateConfigUpdate.IN_PROGRESS)):
    print(".", end="")
    time.sleep(30)

print("\nThe endpoint is ready!", end="")

Agent model version did not have any of the recommended agent signatures. Falling back to checking agent model version compatibility with legacy signatures. Databricks recommends updating and re-logging agents to use the latest signatures; legacy signatures will be removed in the next major MLflow release. See https://docs.databricks.com/en/generative-ai/agent-framework/agent-schema.html for additional details
/local_disk0/.ephemeral_nfs/envs/pythonEnv-dd226c41-d6b5-44be-9eac-da03541bc777/lib/python3.11/site-packages/databricks/agents/utils/mlflow_utils.py:148: FutureWarning: ``mlflow.models.rag_signatures.ChatCompletionRequest`` is deprecated. This method will be removed in a future release. Use ``mlflow.types.llm.ChatCompletionRequest`` instead.
  chat_completions_request_schema = convert_dataclass_to_schema(ChatCompletionRequest())
/local_disk0/.ephemeral_nfs/envs/pythonEnv-dd226c41-d6b5-44be-9eac-da03541bc777/lib/python3.11/site-packages/mlflow/models/rag_signatures.py:25: FutureWa

---------------------------------------------------------------------------
PermissionDenied                          Traceback (most recent call last)
File <command-8080809395929219>, line 8
      5 from databricks import agents
      7 # Deploy the model with the agent framework
----> 8 deployment_info = agents.deploy(
      9     model_name, 
     10     model_version=1,
     11     scale_to_zero=True,
     12     budget_policy_id=None)
     14 # Wait for the Review App and deployed model to be ready
     15 w = WorkspaceClient()

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-dd226c41-d6b5-44be-9eac-da03541bc777/lib/python3.11/site-packages/databricks/agents/deployments.py:808, in deploy(model_name, model_version, scale_to_zero, environment_vars, instance_profile_arn, tags, workload_size, endpoint_name, budget_policy_id, description, deploy_feedback_model, usage_policy_id, **kwargs)
    793 if serving_endpoint_opt is None:
    794     description = description or ""
    795     w.

In [0]:
print(f"Endpoint URL    : {deployment_info.endpoint_url}")
print(f"Review App URL  : {deployment_info.review_app_url}")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8080809395929220>, line 1
----> 1 print(f"Endpoint URL    : {deployment_info.endpoint_url}")
      2 print(f"Review App URL  : {deployment_info.review_app_url}")

NameError: name 'deployment_info' is not defined

## Set Permissions for Other Users

**🚨 Note:** To allow other users for querying and reviewing the app, you need to manually set permissions. To do that;
* Go to **Serving** page and select the deployed endpoint.

* Select **Permissions**.

* Set **Can Query** to "All workspace users".


## 🚨 Mandatory Step: Clean-up

To ensure a smooth workflow, **you must delete the deployed endpoint** before ending the session. This allows other users to deploy a new endpoint without conflicts.  

Run the cell below **before moving forward** to clean up the deployed resources.  

**⚠️ Important:**
- **This step is required** and should be run **before leaving the session**.
- **Instructors should ensure this step is completed** to prevent resource conflicts.


In [0]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
try:
    print("\nCleaning up resources...")
    # Delete endpoint
    agents.delete_deployment(model_name=model_name)
    print(f"Deleted agent endpoint: {model_name}")
    # Delete payload table
    base_table_name = model_name.split(".")[-1]  # rag_app_<suffix>
    payload_table_name = f"{catalog_name}.{schema_name}.{base_table_name}_payload"
    # Drop the table
    spark.sql(f"DROP TABLE IF EXISTS {payload_table_name}")
    print(f"Deleted table: {payload_table_name}")
    # Delete feedback model
    feedback_model_name = f"{catalog_name}.{schema_name}.feedback"
    client.delete_registered_model(name=feedback_model_name)
    print(f"Deleted feedback model: {feedback_model_name}")
except:
    print("An error occured while trying to delete resources. Please try to delete resources manually! Delete these resources: Model Serving Endpoint, Payload Table, and Feedback Model")


Cleaning up resources...
An error occured while trying to delete resources. Please try to delete resources manually! Delete these resources: Model Serving Endpoint, Payload Table, and Feedback Model



## Conclusion

In this demo, we began by defining a custom metric to be used as an additional metric within the Agent Evaluation Framework. Next, we conducted an evaluation run and reviewed the results using both the API and the user interface. In the final step, we deployed the model through Model Serving and demonstrated how the Review App can be utilized to collect human feedback.

&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>